# Offensive Plus-Minus and Defensive Plus-Minus

## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import warnings
from kloppy.domain import EventDataset, Time
from datetime import timedelta

import pandas as pd

from config import players, tournaments

In [3]:
# Ignore specific warnings for cleaner output
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [5]:
# Load players info and team minutes
players_info_df = players.get_players_info(EURO_2024_FINAL_MATCH_ID)
team_minutes_dict = players.get_minutes_played_by_team(EURO_2024_FINAL_MATCH_ID)

In [6]:
# Load match data
dataset, match_events_df = players.load_match_data(EURO_2024_FINAL_MATCH_ID)

## From formula to code

$$PM(m) = \Sigma_{s \in S_{m}} [w(m,s) \times f^{H.RHS}(m,s)]^2 + \Sigma_{s \in S_{m}} [w(m,s) \times f^{A.RHS}(m,s)]^2$$

$$PM(m) = \Sigma_{s \in S_{m}} [w(m,s) \times f^{H.RHS}(m,s)]^2 + \Sigma_{s \in S_{m}} [w(m,s) \times f^{A.RHS}(m,s)]^2$$

### $w(m,s) = w^{TIME}(m,s) \times w^{DURATION}(m,s) \times w^{GOALS}(m,s)$

### $f^{H.RHS}(m,s) = g(m,s,h)$ and $f^{A.RHS}(m,s) = g(m,s,a)$

## Extract match data

In [7]:
def get_match_start_and_end_times(dataset: EventDataset) -> tuple[str, str]:
    """
    Get the start and end times of the match from the dataset.

    Parameters:
    - dataset: The dataset containing match metadata.

    Returns:
        A tuple containing the match start time and end time as strings.
    """
    periods = dataset.metadata.periods
    match_start = str(periods[0].start_time)
    match_end = str(periods[-1].end_time)

    return match_start, match_end

In [8]:
def convert_timedelta_to_minutes(duration: timedelta) -> int:
    """
    Convert the duration of each segment in the match to minutes.

    Parameters:
    - duration: The timedelta object representing the duration.

    Returns:
        The duration in minutes, rounded to the nearest minute.
    """
    minutes, seconds = divmod(duration.seconds, 60)
    return minutes + 1 if seconds >= 30 else minutes

In [9]:
def get_period_duration(dataset: EventDataset) -> dict[str, int]:
    """
    Get the duration of a period in minutes.

    Parameters:
    - dataset: The event dataset containing period information.

    Returns:
        A dictionary with period IDs as keys and their corresponding durations in minutes as values.
    """
    periods = dataset.metadata.periods
    durations = {f"{period.id}": period.duration for period in periods}
    return {f"{key}": convert_timedelta_to_minutes(duration) for key, duration in durations.items()}

In [10]:
def convert_time_str_to_minutes(dataset, time_str: str) -> int:
    """
    Convert a time string in the format "P#T##:##" to minutes.

    Parameters:
    - time_str: The time string to convert.

    Returns:
        The total number of minutes represented by the time string.
    """
    period_durations = get_period_duration(dataset)

    previous_period_minutes = period_durations.get(str(int(time_str[1]) - 1), 0)
    minutes, seconds = map(int, time_str[3:].split(":"))

    return previous_period_minutes + minutes + (1 if seconds >= 30 else 0)

In [11]:
def calculate_segment_duration(dataset, start_time: str, end_time: str) -> int:
    """
    Calculate the duration of a segment in minutes.

    Parameters:
    - dataset: The event dataset containing period information.
    - start_time: The start time of the segment in the format "P#T##:##".
    - end_time: The end time of the segment in the format "P#T##:##".

    Returns:
        The duration of the segment in minutes.
    """
    start_minutes = convert_time_str_to_minutes(dataset, start_time)
    end_minutes = convert_time_str_to_minutes(dataset, end_time)
    duration = end_minutes - start_minutes
    return duration if duration > 0 else 1

In [12]:
def divide_match_in_segments(dataset: EventDataset, match_events_df: pd.DataFrame) -> dict[str, dict]:
    """
    Divide the match into segments based on substititions, red cards, player off and player on events.

    Parameters:
    - dataset: The event dataset containing match information.
    - match_events_df: DataFrame containing match events.

    Returns:
        A dictionary of segments, where each segment contains start time, end time, and goals scored within that segment.
    """
    match_start, match_end = get_match_start_and_end_times(dataset)
    event_times = {match_start, match_end}
    segments = {}

    # Extract event times for substitutions, red cards, player off, and player on events
    substitutions = match_events_df.loc[match_events_df["event_type"] == "SUBSTITUTION", "time"]
    red_cards = match_events_df.loc[
        (match_events_df["event_type"] == "CARD") & (match_events_df["card_type"] == "RED"), "time"
    ]
    player_off = match_events_df.loc[match_events_df["event_type"] == "PLAYER_OFF", "time"]
    player_on = match_events_df.loc[match_events_df["event_type"] == "PLAYER_ON", "time"]

    # Add event times to the set
    event_times.update(substitutions)
    event_times.update(red_cards)
    event_times.update(player_off)
    event_times.update(player_on)

    # Create segments based on event times
    sorted_event_times = sorted(event_times)
    for i in range(len(sorted_event_times) - 1):
        start_time = sorted_event_times[i]
        end_time = sorted_event_times[i + 1]
        segment_key = f"{start_time} - {end_time}"
        segments[segment_key] = {
            "start_time": start_time,
            "end_time": end_time,
            "duration": calculate_segment_duration(dataset, start_time, end_time),
            "goals": match_events_df.loc[
                (match_events_df["time"] >= start_time)
                & (match_events_df["time"] < end_time)
                & (match_events_df["event_type"] == "SHOT")
                & (match_events_df["success"]),
                ["team", "time"],
            ].to_dict(orient="records"),
        }

    return segments

In [13]:
def get_player_start_and_end_times(dataset: EventDataset, player_name: str) -> tuple:
    """
    Determine the start and end times of a player's participation in the match.

    Args:
        dataset: The event dataset containing player position data.
        player_name: The name of the player.

    Returns:
        A tuple containing the start and end times of the player's participation.
    """
    periods_duration = get_period_duration(dataset)

    player_start_time = None
    player_end_time = None

    # Iterate through dataset to find player's position history
    for team in dataset.metadata.teams:
        for player in team.players:
            # Only get position for the specified player
            if player.full_name == player_name:
                for start_time, end_time, _ in player.positions.ranges():
                    if player_start_time is None:
                        player_start_time = start_time
                    if player_end_time is None or end_time > player_end_time:
                        player_end_time = end_time

    # Convert start and end times to minutes, considering previous period durations
    if player_start_time is not None:
        start_minutes_in_period = convert_timedelta_to_minutes(player_start_time.timestamp)
        previous_period_minutes = periods_duration.get(str(int(player_start_time.period.id) - 1), 0)
        player_start_time = start_minutes_in_period + previous_period_minutes
    if player_end_time is not None:
        end_minutes_in_period = convert_timedelta_to_minutes(player_end_time.timestamp)
        previous_period_minutes = periods_duration.get(str(int(player_end_time.period.id) - 1), 0)
        player_end_time = end_minutes_in_period + previous_period_minutes

    return player_start_time, player_end_time

In [14]:
match_segments = divide_match_in_segments(dataset, match_events_df)

In [15]:
RHO_2 = 300.0
RHO_3 = 300.0
RHO_4 = 2.5


def calculate_segments_weight(dataset: EventDataset, match_segments: dict[str, dict]) -> dict[str, dict]:
    """
    Calculate the weight of a segment based on time, duration, and goals.

    Parameters:
    - segment_duration: The duration of the segment in minutes.
    - goal_difference_start: Goals difference in favor of the home team at the start of the segment.
    - goal_difference_end: Goals difference in favor of the home team at the end of the segment.

    Returns:
        The calculated weight for the segment.
    """
    home_team = dataset.metadata.teams[0].name

    # Calculate goal difference at the start and end of the segment
    goal_difference_start = 0

    for segment_key, segment_info in match_segments.items():
        segment_duration = segment_info["duration"]
        goals = segment_info["goals"]

        goal_difference_end = goal_difference_start
        home_goals = 0
        away_goals = 0

        for goal in goals:
            if goal["team"] == home_team:
                goal_difference_end += 1
                home_goals += 1
            else:
                goal_difference_end -= 1
                away_goals += 1

        weight_time = 1.0  # current_date == match_date
        weight_duration = (segment_duration + RHO_2) / RHO_3
        weight_goals = RHO_4 if abs(goal_difference_start) >= 2 and abs(goal_difference_end) >= 2 else 1

        match_segments[segment_key]["weight"] = weight_time * weight_duration * weight_goals
        match_segments[segment_key]["home_goals"] = home_goals
        match_segments[segment_key]["away_goals"] = away_goals

        goal_difference_start = goal_difference_end  # Update for the next segment

    return match_segments

In [16]:
match_segments_data = calculate_segments_weight(dataset, match_segments)

In [17]:
def calculate_plus_minus_score(
    dataset: EventDataset,
    match_segments: dict[str, dict],
    player_name: str,
) -> float:
    """
    Calculate the plus-minus score for a player based on their participation in match segments.

    Parameters:
    - dataset: The event dataset containing match information.
    - match_segments: A dictionary of match segments with their details.
    - player_name: The name of the player.

    Returns:
        The calculated plus-minus score for the player.
    """
    player_start_time, player_end_time = get_player_start_and_end_times(dataset, player_name)

    if player_start_time is None or player_end_time is None:
        return 0.0  # Player did not participate in the match

    plus_minus_score = 0.0

    for segment_info in match_segments.values():
        segment_start_time = convert_time_str_to_minutes(dataset, segment_info["start_time"])
        segment_end_time = convert_time_str_to_minutes(dataset, segment_info["end_time"])

        # Check if the player's participation overlaps with the segment
        if segment_start_time < player_end_time and segment_end_time > player_start_time:
            plus_minus_score += (
                segment_info["weight"] * (segment_info["home_goals"] + segment_info["away_goals"])
            ) ** 2

    return plus_minus_score

In [18]:
def get_match_plus_minus_scores(
    dataset: EventDataset,
    match_segments: dict[str, dict],
    players_info_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Calculate the plus-minus scores for all players in the match.

    Parameters:
    - dataset: The event dataset containing match information.
    - match_segments: A dictionary of match segments with their details.
    - players_info_df: DataFrame containing player information.

    Returns:
        A DataFrame with player names and their corresponding plus-minus scores.
    """

    plus_minus_df = players_info_df.copy()
    for player in players_info_df["player_name"]:
        plus_minus_df.loc[plus_minus_df["player_name"] == player, "plus_minus_score"] = calculate_plus_minus_score(
            dataset, match_segments, player
        )

    plus_minus_df = plus_minus_df.sort_values(by="plus_minus_score", ascending=False).reset_index(drop=True)
    plus_minus_df.insert(0, "rank", range(1, len(plus_minus_df) + 1))  # Add rank column

    for index, row in plus_minus_df.iterrows():
        player_name = row["player_name"]
        player_start_time, player_end_time = get_player_start_and_end_times(dataset, player_name)
        plus_minus_df.at[index, "start_time"] = int(player_start_time)
        plus_minus_df.at[index, "end_time"] = int(player_end_time)

        player_nickname = row["nickname"]
        if pd.notnull(player_nickname):
            plus_minus_df.at[index, "player_name"] = player_nickname

    plus_minus_df.drop(columns=["nickname"], inplace=True)  # Remove the nickname column

    return plus_minus_df

In [19]:
plus_minus_df = get_match_plus_minus_scores(dataset, match_segments_data, players_info_df)
plus_minus_df

,rank,player_name,team_name,minutes_played,plus_minus_score,start_time,end_time
0,1,Lamine Yamal,Spain,90,3.238456,0.0,91.0
1,2,Jordan Pickford,England,96,3.238456,0.0,96.0
2,3,John Stones,England,96,3.238456,0.0,96.0
3,4,Kyle Walker,England,96,3.238456,0.0,96.0
4,5,Jude Bellingham,England,96,3.238456,0.0,96.0
5,6,Nico Williams,Spain,96,3.238456,0.0,96.0
6,7,Martín Zubimendi,Spain,51,3.238456,47.0,96.0
7,8,Bukayo Saka,England,96,3.238456,0.0,96.0
8,9,Marc Cucurella,Spain,96,3.238456,0.0,96.0
9,10,Daniel Olmo,Spain,96,3.238456,0.0,96.0


## Ommited for lack of data

### $f^{H.LHS}(m,s)$ and $f^{A.LHS}(m,s)$

#### $f^{H.SEGMENT}(m, s)$ and $f^{A.SEGMENT}(m, s)$

##### $r(m,s,n)$

In [20]:
NUMBER_OF_RED_CARDS = [1, 2, 3, 4]


def calculate_missing_players_factor(
    missing_players_home: int,
    missing_players_away: int,
) -> dict[int, int]:
    """
    Calculate the missing players factor based on the number of missing players for both teams.

    Parameters:
    - missing_players_home: Number of missing players (red carded + injured without substitution) for the home team.
    - missing_players_away: Number of missing players (red carded + injured without substitution) for the away team.

    Returns:
        A dictionary mapping each red card threshold to its corresponding factor (1, -1, or 0).
    """

    missing_players_factors = dict()

    for n in NUMBER_OF_RED_CARDS:
        if missing_players_home >= n and missing_players_away < n:
            missing_players_factors[n] = 1
        elif missing_players_home < n and missing_players_away >= n:
            missing_players_factors[n] = -1
        else:
            missing_players_factors[n] = 0

    return missing_players_factors

##### $(\beta_n^{O.HRED}$, $\beta_n^{D.HRED}$, $\beta_n^{O.ARED}$, $\beta_n^{D.ARED})$

In [21]:
def calculate_segment_term(
    missing_players_home: int,
    missing_players_away: int,
    weights: dict[int, float],
    neutral_ground_weights: dict[int, float] | None = None,
) -> float:
    """Calculate the segment term based on the number of missing players."""
    total = 0.0
    missing_players_factors = calculate_missing_players_factor(missing_players_home, missing_players_away)

    for n in NUMBER_OF_RED_CARDS:
        if neutral_ground_weights is None:
            total += missing_players_factors[n] * weights[n]
        else:
            total += missing_players_factors[n] * ((weights[n] + neutral_ground_weights[n]) / 2)

    return total

In [22]:
# This factors are calculated using a gradiant descent search. We don't have them.
RED_FACTOR_OFFENSIVE_HOME = {1: 0.1, 2: 0.2, 3: 0.3, 4: 0.4}
RED_FACTOR_DEFENSIVE_HOME = {1: 0.05, 2: 0.1, 3: 0.15, 4: 0.2}
RED_FACTOR_OFFENSIVE_AWAY = {1: 0.08, 2: 0.16, 3: 0.24, 4: 0.32}
RED_FACTOR_DEFENSIVE_AWAY = {1: 0.04, 2: 0.08, 3: 0.12, 4: 0.16}


def calculate_home_away_segment_terms(
    red_cards_home: int,
    red_cards_away: int,
    injured_players_home: int,
    injured_players_away: int,
    neutral_ground: bool = False,
) -> tuple[float, float]:
    """
    Calculate the segment factors for offensive and defensive situations based on red cards and injured players.

    Parameters:
    - red_cards_home: Number of red cards for the home team.
    - red_cards_away: Number of red cards for the away team.
    - injured_players_home: Number of injured players for the home team.
    - injured_players_away: Number of injured players for the away team.
    - neutral_ground: Boolean indicating if the match is played on neutral ground.

    Returns:
        A tuple containing the segment factors for the home and away teams.
    """
    missing_players_home = red_cards_home + injured_players_home
    missing_players_away = red_cards_away + injured_players_away

    # Calculate the red card advantage based on the number of missing players for both teams
    advantage = sum(calculate_missing_players_factor(missing_players_home, missing_players_away).values())

    # Determine the segment factor based on the home/away status, advantage, and whether the match is on neutral ground
    factor_segment_home = 0.0
    factor_segment_away = 0.0

    if not neutral_ground and advantage >= 0:
        factor_segment_home = -calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_OFFENSIVE_HOME,
        )
        factor_segment_away = calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_DEFENSIVE_HOME,
        )

    elif not neutral_ground and advantage < 0:
        factor_segment_home = -calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_DEFENSIVE_AWAY,
        )
        factor_segment_away = calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_OFFENSIVE_AWAY,
        )

    elif neutral_ground and advantage >= 0:
        factor_segment_home = -calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_OFFENSIVE_HOME,
            RED_FACTOR_OFFENSIVE_AWAY,
        )
        factor_segment_away = calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_DEFENSIVE_HOME,
            RED_FACTOR_DEFENSIVE_AWAY,
        )

    elif neutral_ground and advantage < 0:
        factor_segment_home = -calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_DEFENSIVE_AWAY,
            RED_FACTOR_DEFENSIVE_HOME,
        )
        factor_segment_away = calculate_segment_term(
            missing_players_home,
            missing_players_away,
            RED_FACTOR_OFFENSIVE_AWAY,
            RED_FACTOR_OFFENSIVE_HOME,
        )

    return factor_segment_home, factor_segment_away

#### $f^{H.MATCH}(m, s)$ and $f^{A.MATCH}(m, s)$

In [23]:
def calculate_home_away_match_terms(
    neutral_ground: bool = False,
) -> tuple[float, float]:
    """Calculate the match factors for home and away teams based on offensive and defensive home-field advantage."""
    # This factors are calculated using a gradient descent search. We don't have them.
    FACTOR_OFFENSIVE_HFA = 0.3
    FACTOR_DEFENSIVE_HFA = 0.2

    if neutral_ground:
        factor_match_home = 0.0
        factor_match_away = 0.0
    else:
        factor_match_home = FACTOR_OFFENSIVE_HFA
        factor_match_away = -FACTOR_DEFENSIVE_HFA

    return factor_match_home, factor_match_away

#### $f^{O.PLAYER}(m,s,p)$ and $f^{D.PLAYER}(m,s,p)$

In [24]:
Y_MIN = 16
Y_MAX = 42
Y_SET = range(Y_MIN, Y_MAX + 1)

# The date of birth of players is not available in the StatsBomb dataset
# player_age = match_date - player_birth_date


def calculate_age_weight_for_match(player_age: float) -> dict[int, float]:
    """Calculate the weight for each age segment based on the player's age."""
    u_weights = {}

    for i, y in enumerate(Y_SET, start=1):
        if y == Y_MIN and player_age <= Y_MIN:
            u_weights[i] = 1
        elif y == Y_MAX and player_age >= Y_MAX:
            u_weights[i] = 1
        elif Y_SET[i - 1] <= player_age <= y:
            u_weights[i] = (player_age - y) / (Y_SET[i + 1] - Y_SET[i])
        elif y <= player_age <= Y_SET[i + 1]:
            u_weights[i] = (Y_SET[i + 1] - player_age) / (Y_SET[i + 1] - Y_SET[i])
        else:
            u_weights[i] = 0

    return u_weights

In [25]:
# This factors are calculated using a gradient descent search. We don't have them.
BETA_OFFENSIVE = {"player1": 0.5, "player2": 0.3, "player3": 0.8}
BETA_DEFENSIVE = {"player1": 0.4, "player2": 0.6, "player3": 0.2}


def calculate_offensive_and_deffensive_player_terms(player: str) -> tuple[float, float]:
    """
    Calculate the offensive and defensive terms for a given player.

    Parameters:
    - player: The name of the player.

    Returns:
        A tuple containing the offensive and defensive terms for the player.
    """

    factor_offensive_player = BETA_OFFENSIVE.get(player, 0)
    factor_defensive_player = BETA_DEFENSIVE.get(player, 0)

    return factor_offensive_player, factor_defensive_player